# Full 5 within-dataset classical baselines

Notebook này chạy đủ 5 dataset trong paper: `CCLE`, `CTRPv2`, `GCSI/gCSI`, `GDSCv1`, `GDSCv2`. Không làm cross-dataset.

Vì PyTDC `DrugRes` trong env `demo` chỉ support `gdsc1/gdsc2`, 5-dataset raw data được lấy từ CSA benchmark của paper. Sau khi load raw data, notebook dựng dataframe theo cùng schema với `data_eda_pca_preparation.ipynb`: `Drug_ID`, `Drug`, `Cell Line_ID`, `Cell Line`, `Y`, rồi preprocess y hệt: cold split theo `Cell Line_ID`, Morgan fingerprint 1024 bits cho drug, `StandardScaler + PCA(0.95)` fit trên train cell lines only.

In [1]:
from pathlib import Path
import gc
import json
import os
import time
import urllib.request
import warnings
import zipfile

CACHE_DIR = Path('../.cache')
(CACHE_DIR / 'matplotlib').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'fontconfig').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str((CACHE_DIR / 'matplotlib').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR.resolve()))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 160)


In [2]:
RANDOM_STATE = 42
MORGAN_BITS = 1024
PCA_VARIANCE = 0.95
SPLIT_FRAC = [0.7, 0.1, 0.2]
TRAIN_FRACTIONS = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

DATASETS = {
    'CCLE': ['CCLE'],
    'CTRPv2': ['CTRPv2', 'CTRP'],
    'GCSI': ['GCSI', 'gCSI'],
    'GDSCv1': ['GDSCv1', 'GDSC1'],
    'GDSCv2': ['GDSCv2', 'GDSC2'],
}

DATA_DIR = Path('../data')
CSA_ZIP_URL = 'https://zenodo.org/records/15258883/files/csa_data.zip?download=1'
CSA_ZIP_PATH = DATA_DIR / 'csa_data.zip'
CSA_RAW_DIR = DATA_DIR / 'csa_data' / 'raw_data'

PROCESSED_DIR = Path('../processed_data/model_ready_five_within')
ARTIFACT_DIR = PROCESSED_DIR / 'artifacts'
OUTPUT_DIR = Path('../models/five_dataset_within_learning_curves')

for path in [DATA_DIR, PROCESSED_DIR, ARTIFACT_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

models = {
    'ridge': Ridge(alpha=100.0),
    'lasso': Lasso(alpha=0.001, max_iter=5000, tol=1e-3, selection='random', random_state=RANDOM_STATE),
    'random_forest': RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features='sqrt',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    'knn': KNeighborsRegressor(n_neighbors=31, weights='distance', metric='minkowski', p=2, n_jobs=-1),
}

print('Datasets:', list(DATASETS))
print('Processed output:', PROCESSED_DIR.resolve())
print('Model output:', OUTPUT_DIR.resolve())


Datasets: ['CCLE', 'CTRPv2', 'GCSI', 'GDSCv1', 'GDSCv2']
Processed output: /Users/vietanh/Desktop/ML-Predicting-drug-response/processed_data/model_ready_five_within
Model output: /Users/vietanh/Desktop/ML-Predicting-drug-response/models/five_dataset_within_learning_curves


In [3]:
def ensure_csa_data():
    required_files = [
        CSA_RAW_DIR / 'y_data' / 'response.tsv',
        CSA_RAW_DIR / 'x_data' / 'drug_SMILES.tsv',
        CSA_RAW_DIR / 'x_data' / 'cancer_gene_expression.tsv',
    ]
    if all(path.exists() for path in required_files):
        return

    if not CSA_ZIP_PATH.exists():
        print(f'Downloading CSA benchmark data to {CSA_ZIP_PATH} ...')
        urllib.request.urlretrieve(CSA_ZIP_URL, CSA_ZIP_PATH)

    print(f'Extracting {CSA_ZIP_PATH} ...')
    with zipfile.ZipFile(CSA_ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)

    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing CSA files after extraction: ' + ', '.join(missing))


def load_raw_tables():
    ensure_csa_data()
    response = pd.read_csv(CSA_RAW_DIR / 'y_data' / 'response.tsv', sep='\t')
    smiles = pd.read_csv(CSA_RAW_DIR / 'x_data' / 'drug_SMILES.tsv', sep='\t')
    gene_expression = pd.read_csv(CSA_RAW_DIR / 'x_data' / 'cancer_gene_expression.tsv', sep='\t', index_col=0)
    gene_expression = gene_expression.astype(np.float32)
    smiles_map = smiles.set_index('improve_chem_id')['canSMILES'].to_dict()

    print('response:', response.shape)
    print('drug smiles:', smiles.shape)
    print('gene expression:', gene_expression.shape)
    print('response sources:', sorted(response['source'].dropna().unique()))
    return response, smiles_map, gene_expression


def pick_source_name(response, aliases):
    available = set(response['source'].dropna().unique())
    for alias in aliases:
        if alias in available:
            return alias
    raise ValueError(f'None of aliases {aliases} found. Available sources: {sorted(available)}')


In [4]:
def smiles_to_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(MORGAN_BITS, dtype=np.int8)
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=MORGAN_BITS), dtype=np.int8)
    except Exception:
        return np.zeros(MORGAN_BITS, dtype=np.int8)


def cold_split_by_cell_line(target_df):
    rng = np.random.default_rng(RANDOM_STATE)
    unique_cells = target_df['Cell Line_ID'].drop_duplicates().to_numpy()
    unique_cells = rng.permutation(unique_cells)

    n_train = int(len(unique_cells) * SPLIT_FRAC[0])
    n_val = int(len(unique_cells) * SPLIT_FRAC[1])
    train_cells = set(unique_cells[:n_train])
    val_cells = set(unique_cells[n_train:n_train + n_val])
    test_cells = set(unique_cells[n_train + n_val:])

    train_df = target_df[target_df['Cell Line_ID'].isin(train_cells)].reset_index(drop=True)
    val_df = target_df[target_df['Cell Line_ID'].isin(val_cells)].reset_index(drop=True)
    test_df = target_df[target_df['Cell Line_ID'].isin(test_cells)].reset_index(drop=True)
    return train_df, val_df, test_df


def build_dataset(target_df, pca_map):
    drug_feats = np.array([smiles_to_fp(s) for s in target_df['Drug'].values], dtype=np.int8)
    cell_feats = np.array([pca_map[cid] for cid in target_df['Cell Line_ID'].values], dtype=np.float32)
    X = np.hstack([drug_feats, cell_feats]).astype(np.float32)
    y = target_df['Y'].values.astype(np.float32)

    del drug_feats, cell_feats
    gc.collect()
    return X, y


def make_model_ready_dataframe(dataset_name, aliases, response, smiles_map, gene_expression):
    source_name = pick_source_name(response, aliases)
    df = response.loc[
        response['source'] == source_name,
        ['improve_chem_id', 'improve_sample_id', 'auc'],
    ].copy()
    df = df.rename(columns={
        'improve_chem_id': 'Drug_ID',
        'improve_sample_id': 'Cell Line_ID',
        'auc': 'Y',
    })
    df = df.dropna(subset=['Drug_ID', 'Cell Line_ID', 'Y'])
    df = df[df['Drug_ID'].isin(smiles_map)]
    df = df[df['Cell Line_ID'].isin(gene_expression.index)]
    df = df.reset_index(drop=True)

    df['Drug'] = df['Drug_ID'].map(smiles_map)
    cell_vector_map = {cid: gene_expression.loc[cid].to_numpy(dtype=np.float32) for cid in df['Cell Line_ID'].drop_duplicates()}
    df['Cell Line'] = df['Cell Line_ID'].map(cell_vector_map)
    df = df[['Drug_ID', 'Drug', 'Cell Line_ID', 'Cell Line', 'Y']]
    return source_name, df


def prepare_dataset(dataset_name, aliases, response, smiles_map, gene_expression):
    source_name, df = make_model_ready_dataframe(dataset_name, aliases, response, smiles_map, gene_expression)
    train_df, val_df, test_df = cold_split_by_cell_line(df)

    train_unique_cells = train_df.drop_duplicates(subset=['Cell Line_ID'])
    train_matrix = np.array([np.array(val, dtype=np.float32) for val in train_unique_cells['Cell Line'].values])

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_matrix)

    pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
    pca.fit(train_scaled)

    all_unique_df = df.drop_duplicates(subset=['Cell Line_ID'])
    all_ids = all_unique_df['Cell Line_ID'].values
    all_matrix = np.array([np.array(val, dtype=np.float32) for val in all_unique_df['Cell Line'].values])
    all_pca_feats = pca.transform(scaler.transform(all_matrix))
    cell_pca_map = {cid: feat for cid, feat in zip(all_ids, all_pca_feats)}

    print(f'{dataset_name}: source={source_name}, raw={df.shape}')
    print(f'{dataset_name}: train={train_df.shape}, val={val_df.shape}, test={test_df.shape}')
    print(f'{dataset_name}: PCA components={pca.n_components_}, explained={np.sum(pca.explained_variance_ratio_):.2%}')

    X_train, y_train = build_dataset(train_df, cell_pca_map)
    X_val, y_val = build_dataset(val_df, cell_pca_map)
    X_test, y_test = build_dataset(test_df, cell_pca_map)

    dataset_key = dataset_name.lower().replace('v', 'v')
    dataset_dir = PROCESSED_DIR / dataset_key
    artifact_dir = ARTIFACT_DIR / dataset_key
    dataset_dir.mkdir(parents=True, exist_ok=True)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    npz_path = dataset_dir / f'{dataset_key}_lgbm_style_data.npz'
    np.savez_compressed(
        npz_path,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
    )
    joblib.dump(scaler, artifact_dir / 'cell_line_scaler.joblib')
    joblib.dump(pca, artifact_dir / 'cell_line_pca.joblib')

    metadata = {
        'dataset': dataset_name,
        'source_name': source_name,
        'data_source': 'CSA benchmark from paper, converted to DrugRes-like dataframe',
        'split_method': 'cold_split by Cell Line_ID',
        'split_frac': SPLIT_FRAC,
        'random_state': RANDOM_STATE,
        'drug_feature': f'Morgan fingerprint {MORGAN_BITS} bits',
        'cell_feature': 'StandardScaler + PCA fitted on train cell lines only',
        'cell_pca_components': int(pca.n_components_),
        'raw_shape': list(df.shape),
        'X_train_shape': list(X_train.shape),
        'X_val_shape': list(X_val.shape),
        'X_test_shape': list(X_test.shape),
        'saved_npz': str(npz_path),
    }
    with open(artifact_dir / 'metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    del df, train_df, val_df, test_df, train_matrix, train_scaled, all_unique_df, all_matrix, all_pca_feats, cell_pca_map
    gc.collect()
    return dataset_key, metadata, (X_train, y_train, X_val, y_val, X_test, y_test)


In [5]:
def safe_corr(y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def compute_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'pearson': safe_corr(y_true, y_pred),
    }


def run_learning_curve(dataset_name, model_name, base_model, arrays):
    X_train, y_train, X_val, y_val, X_test, y_test = arrays
    rng = np.random.default_rng(RANDOM_STATE)
    train_perm = rng.permutation(len(y_train))
    rows = []

    for fraction in TRAIN_FRACTIONS:
        sample_size = max(20, int(len(y_train) * fraction))
        sample_idx = train_perm[:sample_size]

        model = clone(base_model)
        start = time.time()
        model.fit(X_train[sample_idx], y_train[sample_idx])
        fit_seconds = time.time() - start

        train_pred = model.predict(X_train[sample_idx])
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'train_fraction': fraction,
            'train_size': int(sample_size),
            'val_size': int(len(y_val)),
            'test_size': int(len(y_test)),
            'fit_seconds': float(fit_seconds),
            **{f'train_{k}': v for k, v in compute_metrics(y_train[sample_idx], train_pred).items()},
            **{f'val_{k}': v for k, v in compute_metrics(y_val, val_pred).items()},
            **{f'test_{k}': v for k, v in compute_metrics(y_test, test_pred).items()},
        })

        del model, train_pred, val_pred, test_pred
        gc.collect()

    return pd.DataFrame(rows)


def fit_full_train_eval_test(dataset_name, dataset_key, model_name, base_model, arrays):
    X_train, y_train, X_val, y_val, X_test, y_test = arrays
    model_dir = OUTPUT_DIR / dataset_key
    model_dir.mkdir(parents=True, exist_ok=True)

    model = clone(base_model)
    start = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    row = {
        'dataset': dataset_name,
        'model': model_name,
        'train_size': int(len(y_train)),
        'val_size': int(len(y_val)),
        'test_size': int(len(y_test)),
        'fit_seconds': float(fit_seconds),
        **{f'train_{k}': v for k, v in compute_metrics(y_train, train_pred).items()},
        **{f'val_{k}': v for k, v in compute_metrics(y_val, val_pred).items()},
        **{f'test_{k}': v for k, v in compute_metrics(y_test, test_pred).items()},
    }

    joblib.dump(model, model_dir / f'{model_name}.joblib')
    np.save(model_dir / f'{model_name}_val_pred.npy', val_pred)
    np.save(model_dir / f'{model_name}_test_pred.npy', test_pred)
    return row


In [6]:
response, smiles_map, gene_expression = load_raw_tables()

all_metadata = []
all_learning_curves = []
all_full_eval = []

for dataset_name, aliases in DATASETS.items():
    print('\n' + '=' * 90)
    print('Dataset:', dataset_name)
    dataset_key, metadata, arrays = prepare_dataset(dataset_name, aliases, response, smiles_map, gene_expression)
    all_metadata.append(metadata)

    curve_parts = []
    full_eval_rows = []
    for model_name, model in models.items():
        print('Learning curve for', dataset_name, model_name)
        curve_parts.append(run_learning_curve(dataset_name, model_name, model, arrays))

        print('Full train/test eval for', dataset_name, model_name)
        full_eval_rows.append(fit_full_train_eval_test(dataset_name, dataset_key, model_name, model, arrays))

    learning_curve_df = pd.concat(curve_parts, ignore_index=True)
    full_eval_df = pd.DataFrame(full_eval_rows).sort_values('val_rmse')

    dataset_output_dir = OUTPUT_DIR / dataset_key
    learning_curve_df.to_csv(dataset_output_dir / 'best_fixed_models_learning_curves.csv', index=False)
    full_eval_df.to_csv(dataset_output_dir / 'best_fixed_models_full_eval.csv', index=False)

    display(learning_curve_df.round(4))
    display(full_eval_df.round(4))

    all_learning_curves.append(learning_curve_df)
    all_full_eval.append(full_eval_df)
    del arrays, curve_parts, full_eval_rows, learning_curve_df, full_eval_df
    gc.collect()

metadata_df = pd.DataFrame(all_metadata)
learning_curves_all_df = pd.concat(all_learning_curves, ignore_index=True)
full_eval_all_df = pd.concat(all_full_eval, ignore_index=True).sort_values(['dataset', 'val_rmse'])

metadata_df.to_csv(PROCESSED_DIR / 'metadata_all_datasets.csv', index=False)
learning_curves_all_df.to_csv(OUTPUT_DIR / 'best_fixed_models_learning_curves_all_datasets.csv', index=False)
full_eval_all_df.to_csv(OUTPUT_DIR / 'best_fixed_models_full_eval_all_datasets.csv', index=False)

display(metadata_df)
display(full_eval_all_df.round(4))


Extracting ../data/csa_data.zip ...


ValueError: could not convert string to float: 'TSPAN6'

In [ ]:
best_by_dataset = full_eval_all_df.loc[full_eval_all_df.groupby('dataset')['val_rmse'].idxmin()].sort_values('val_rmse')
display(best_by_dataset.round(4))
best_by_dataset.to_csv(OUTPUT_DIR / 'best_model_by_dataset.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=full_eval_all_df, x='dataset', y='val_rmse', hue='model', ax=axes[0])
sns.barplot(data=full_eval_all_df, x='dataset', y='test_rmse', hue='model', ax=axes[1])

axes[0].set_title('Full-train Validation RMSE')
axes[1].set_title('Full-train Test RMSE')
for ax in axes:
    ax.set_xlabel('Dataset')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'full_eval_rmse_by_dataset.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
models_order = list(models.keys())
colors = plt.cm.tab10(np.linspace(0, 1, len(models_order)))
color_map = dict(zip(models_order, colors))

for dataset_name in DATASETS:
    dataset_key = dataset_name.lower().replace('v', 'v')
    plot_df = learning_curves_all_df[learning_curves_all_df['dataset'] == dataset_name]
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    plot_specs = [
        (axes[0, 0], 'train_rmse', 'Train RMSE', 'RMSE'),
        (axes[0, 1], 'val_rmse', 'Validation RMSE', 'RMSE'),
        (axes[0, 2], 'test_rmse', 'Test RMSE', 'RMSE'),
        (axes[1, 0], 'train_r2', 'Train R2', 'R2'),
        (axes[1, 1], 'val_r2', 'Validation R2', 'R2'),
        (axes[1, 2], 'test_r2', 'Test R2', 'R2'),
    ]

    for model_name in models_order:
        model_df = plot_df[plot_df['model'] == model_name].sort_values('train_size')
        for ax, metric_col, _, _ in plot_specs:
            ax.plot(model_df['train_size'], model_df[metric_col], marker='o', linewidth=2, label=model_name, color=color_map[model_name])

    for ax, _, title, ylabel in plot_specs:
        ax.set_title(title)
        ax.set_xlabel('Training samples')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=4, frameon=True)
    fig.suptitle(f'Learning Curves - {dataset_name}', fontsize=18, weight='bold')
    plt.tight_layout(rect=[0, 0.08, 1, 0.96])
    fig.savefig(OUTPUT_DIR / dataset_key / 'best_fixed_models_learning_curves_big_plot.png', dpi=180, bbox_inches='tight')
    plt.show()

print('Saved results to:', OUTPUT_DIR.resolve())
print('Saved processed data to:', PROCESSED_DIR.resolve())
